## 2026-05-19

### 训练方向和内容总结

1. 双均线计算

使用 ```rolling(window=n).mean()``` 计算移动平均线
MA5（短期）、MA20（长期）

2. 交易信号生成

根据 MA5 与 MA20 的大小关系生成持仓信号（Signal）
使用 ```diff()``` 捕捉信号变化点，得到买卖点（Position）

3. 买卖点标记与可视化

用 ```pd.Series``` + 索引匹配的方式生成买卖点标记
使用``` mplfinance``` 的``` make_addplot``` 叠加均线和标记点

4. 策略收益计算

根据前一天的持仓状态乘当日收益率计算策略每日收益
累积收益、最大回撤、年化收益率、胜率等指标
5. K线图与均线叠加

```mpf.plot``` + ```make_addplot``` 实现多图层绘制

In [ ]:
# 填空题

# 1.计算5日均线的代码是：
# data['MA5'] = data['Close'].rolling(window=__1__).mean()
data['MA5'] = data['Close'].rolling(window=5).mean()

# 2.当 MA5 > MA20 时，将 Signal 列设为 2，否则保持为 0。
data['Signal'] = 0
data.loc[data['MA5'] > data['MA20'], 'Signal'] = 2

# 3.捕捉持仓信号变化的函数是 data['Position'] = data['Signal'].__3__()
data['Position'] = data['Signal'].diff()

# 4.买入信号对应的 Position 值为 4，卖出信号为 5。
buy_signal = data[data['Position'] == 4]
sell_signal = data[data['Position'] == 5]

# 5.在 mplfinance 中，向 K 线图添加均线的函数是 mpf.make_addplot，其类型参数 type 用于散点图时值为 '__6__'。
# mpf.make_addplot，其类型参数 type 用于散点图时值为 'apds'。?什么类型，有什么意义？

# 6.策略每日收益率的计算方式是：data['Strategy_Returns'] = data['Position_Holdings'].__7__(1) * data['Returns']
data['Strategy_Returns'] = data['Position_Holdings'].shift(1) * data['Returns']  #shift是什么？怎么用？

# 7.累积收益的计算公式是：(1 + data['Returns']).__8__()
(1 + data['Returns']).cumprod()  #cumprod是什么？怎么用

# 8.最大回撤的计算中，running_max = cumulative.__9__().max()
running_max = cumulative.expanding().max()  #整个都看不懂


# 造句题

#需求1
#在现有的双均线策略基础上，增加一个过滤条件：只有当收盘价 > MA20 时才开仓（即 MA5 > MA20 且 Close > MA20 才买入）。
    # 一、生成交易信号
data['Signal'] = 0
data.loc[data['MA5'] > data['MA20'] & data['Close'] > data['MA20'], 'Signal'] = 1
data['Position'] = data['Signal'].diff()
    # 二、找出具体买卖点
buy_signals = data[data['Position'] == 1]
sell_signals = data[data['Position'] == -1]



#需求2
#计算策略的夏普比率（假设无风险利率为 0），只用一行或两行代码。
# 计算每笔交易的盈亏
trade_returns = []
for i in range(len(buy_signals_correct)):
    buy_date = buy_signals_correct.index[i]
    # 找到对应的卖出日期
    if i < len(sell_signals_correct):
        sell_date = sell_signals_correct.index[i]
        trade_return = (data.loc[sell_date, 'Cumulative_Strategy'] /
                       data.loc[buy_date, 'Cumulative_Strategy'] - 1)
        trade_returns.append(trade_return)
# 年化收益率（假设252个交易日）
years = total_days / 252
annual_return = (data['Cumulative_Strategy'].iloc[-1] ** (1/years) - 1) if years > 0 else 0
# 夏普比率
Sharpe Ratio = (annual_return - 0.0000986) / (std(annual_return) * math.sqrt(252))



#需求3
#用 matplotlib 单独画一张图：策略累计收益 vs 买入持有累计收益，并用 fill_between 填充两者之间的差值区域。
import matplotlib.pyplot as plt
import numpy as np


#需求4
#在现有 K 线图上，用 axvspan 标注出所有持仓区间（即 MA5 > MA20 的区间）。
# axvspan标记部分
y1 = data['MA5']
y2 = data['MA20']

higher = y1 > y2



In [1]:
# ============================================
# 以下为修正版（只写改错的题）
# ============================================

# 填空题
# 5.在 mplfinance 中，向 K 线图添加均线的函数是 mpf.make_addplot，其类型参数 type 用于散点图时值为 '__6__'。
# mpf.make_addplot，其类型参数 type 用于散点图时值为 'scatter'。
    # 错因分析：apds是 add plots的缩写，只是一个变量名；而scatter是mpf.make_addplot()函数中type参数的一个可选值，表示要添加的图形是散点图。

# 造句题
#需求1
#在现有的双均线策略基础上，增加一个过滤条件：只有当收盘价 > MA20 时才开仓（即 MA5 > MA20 且 Close > MA20 才买入）。
# 原答案：data.loc[data['MA5'] > data['MA20'] & data['Close'] > data['MA20'], 'Signal'] = 1
data.loc[(data['MA5'] > data['MA20']) & (data['Close'] > data['MA20']), 'Signal'] = 1
# 运算符的优先级来看，& > >,（条件1） & （条件2）才是正确的逻辑

#需求2
#计算策略的夏普比率（假设无风险利率为 0），只用一行或两行代码。
daily_returns = data['Strategy_Returns'].dropna()
sharpe_ratio = daily_returns.mean() / daily_returns.std() * (252 ** 0.5)
# Sharpe Ratio = (Rp - Rf) / σp
# Rp：策略收益率（均值）  Rf：无风险收益率   σp：策略收益率的标准差（风险）
# Sharpe年化 =  Sharpe周期数 * （（周期数））** 0.5

#需求3
#用 matplotlib 单独画一张图：策略累计收益 vs 买入持有累计收益，并用 fill_between 填充两者之间的差值区域。
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))     # 设置画布（12英寸 * 5英寸）
plt.plot(data.index, data['Cumulative_Market'], label='Buy & Hold')
plt.plot(data.index, data['Cumulative_Strategy'], label='MA Strategy')
plt.legend()  # 把label显示到图上
plt.show()
# 主要画线语句
# plt.plot(x轴，y轴，其他参数)，在这里就是plt.plot(日期，数据，其他参数)



#需求4
#在现有 K 线图上，用 axvspan 标注出所有持仓区间（即 MA5 > MA20 的区间）。
# 假设原来的 apds 是：
apds = [
    mpf.make_addplot(data['MA5'], color='blue'),
    mpf.make_addplot(data['MA20'], color='orange'),
    mpf.make_addplot(buy_markers, type='scatter', markersize=100, marker='^', color='green'),
    mpf.make_addplot(sell_markers, type='scatter', markersize=100, marker='v', color='red')
]

# 只需要加这一行：
apds.append(mpf.make_addplot(background, type='bar', alpha=0.2, color='gray', panel=0, width=0.8))
# 代码实现思路：在每个MA5 > MA20的日期区间画一个柱子，设置成半透明，就没挡住k线了

NameError: name 'data' is not defined